In [0]:
%run /Workspace/Users/felipemkautzmann@gmail.com/vatenfall-simulator/src/transforms/silver/reference_transform

In [0]:
# ============================================
# 1. IMPORTS
# ============================================
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, regexp_replace, regexp_extract, trim, split, when, size

# ============================================
# 2. YOUR FUNCTIONS (KEEP YOUR ORIGINAL VERSIONS)
# ============================================
def standarize_asset_reference(df: DataFrame) -> DataFrame:
    return (df
        .withColumn("asset_name", 
                    regexp_replace("asset_name", "^Asset ", "")
        ))

def standarize_region_reference(df: DataFrame) -> DataFrame:
    return (df
        .withColumn("operating_zone", regexp_extract(col("operating_zone"), "\\d+", 0))
        .withColumn("region", trim(col("region")))
        .withColumn("country_code", trim(col("country_code")))
    )

def standarize_event_type_reference(df: DataFrame) -> DataFrame:
    return (df
        .withColumn('event_type', trim(col('event_type')))
        .withColumn('category', trim(col('category')))
    )

# FIXED VERSION (use this to avoid crash)
def standarize_market_references_fixed(df: DataFrame) -> DataFrame:
    return (df
        .withColumn('split_temp', split(col('description'), " ", 2))
        .withColumn('description', 
                    when(size(col('split_temp')) > 1, 
                         trim(col('split_temp').getItem(1)))
                    .otherwise(None))
        .withColumn('market_type', trim(col('market_type')))
        .drop('split_temp')
    )

def standarize_weather_alert_reference(df: DataFrame) -> DataFrame:
    return (df
        .withColumn('weather_alert_level', trim(col('weather_alert_level')))
    )

# ============================================
# 3. TESTS
# ============================================
def test_standarize_asset_reference():
    print("🧪 Test 1: standarize_asset_reference")
    
    test_data = [("Asset Turbine 42",), ("Asset ",), ("Asset",), ("Substation Asset A",), ("NoPrefix",), (None,), ("",)]
    schema = StructType([StructField("asset_name", StringType())])
    df = spark.createDataFrame(test_data, schema)
    result = standarize_asset_reference(df)
    results = result.collect()
    
    assert results[0]["asset_name"] == "Turbine 42"
    assert results[1]["asset_name"] == ""
    assert results[2]["asset_name"] == "Asset"
    assert results[3]["asset_name"] == "Substation Asset A"
    assert results[4]["asset_name"] == "NoPrefix"
    assert results[5]["asset_name"] is None
    assert results[6]["asset_name"] == ""
    
    print("✅ test_standarize_asset_reference PASSED\n")

def test_standarize_region_reference():
    print("🧪 Test 2: standarize_region_reference")
    
    test_data = [("Zone42", "  Berlin  ", "  DE  "), ("Zone7B", "Frankfurt", "FR"), ("North", "Munich", "DE"), ("Zone12345", "  Hamburg  ", "  "), ("", "Cologne", "DE"), (None, "Berlin", "DE"), ("Zone1", None, "DE"), ("Zone2", "Berlin", None)]
    schema = StructType([StructField("operating_zone", StringType()), StructField("region", StringType()), StructField("country_code", StringType())])
    df = spark.createDataFrame(test_data, schema)
    result = standarize_region_reference(df)
    results = result.collect()
    
    assert results[0]["operating_zone"] == "42"
    assert results[0]["region"] == "Berlin"
    assert results[0]["country_code"] == "DE"
    assert results[1]["operating_zone"] == "7"
    assert results[2]["operating_zone"] == ""
    assert results[3]["operating_zone"] == "12345"
    assert results[4]["operating_zone"] == ""
    assert results[5]["operating_zone"] is None
    assert results[6]["region"] is None
    assert results[7]["country_code"] is None
    
    print("✅ test_standarize_region_reference PASSED\n")

def test_standarize_event_type_reference():
    print("🧪 Test 3: standarize_event_type_reference")
    
    test_data = [("  ERROR  ", "  CRITICAL  "), ("INFO", "LOW"), ("  ", "  "), ("", ""), (None, "HIGH"), ("WARNING", None)]
    schema = StructType([StructField("event_type", StringType()), StructField("category", StringType())])
    df = spark.createDataFrame(test_data, schema)
    result = standarize_event_type_reference(df)
    results = result.collect()
    
    assert results[0]["event_type"] == "ERROR"
    assert results[0]["category"] == "CRITICAL"
    assert results[1]["event_type"] == "INFO"
    assert results[2]["event_type"] == ""
    assert results[2]["category"] == ""
    assert results[3]["event_type"] == ""
    assert results[4]["event_type"] is None
    assert results[5]["category"] is None
    
    print("✅ test_standarize_event_type_reference PASSED\n")

def test_standarize_weather_alert_reference():
    print("🧪 Test 4: standarize_weather_alert_reference")
    
    test_data = [("  RED  ",), ("YELLOW",), ("  ",), ("",), (None,)]
    schema = StructType([StructField("weather_alert_level", StringType())])
    df = spark.createDataFrame(test_data, schema)
    result = standarize_weather_alert_reference(df)
    results = result.collect()
    
    assert results[0]["weather_alert_level"] == "RED"
    assert results[1]["weather_alert_level"] == "YELLOW"
    assert results[2]["weather_alert_level"] == ""
    assert results[3]["weather_alert_level"] == ""
    assert results[4]["weather_alert_level"] is None
    
    print("✅ test_standarize_weather_alert_reference PASSED\n")

def run_all_tests():
    print("\n" + "="*60)
    print("🎯 RUNNING ALL TESTS")
    print("="*60 + "\n")
    
    test_standarize_asset_reference()
    test_standarize_region_reference()
    test_standarize_event_type_reference()
    test_standarize_weather_alert_reference()
    
    print("="*60)
    print("🎉 ALL TESTS PASSED!")
    print("="*60)

# ============================================
# 4. RUN TESTS
# ============================================
run_all_tests()